In [2]:
import yaml
import torch
import numpy as np
from torch_geometric.loader import DataLoader
import pandas as pd
from model import ILBERT_T
from dataset import SMILES_dataset
from ILtokenizer import SMILES_Atomwise_Tokenizer
from torch.utils.data import DataLoader
from tqdm import tqdm
from torch.cuda.amp import autocast
from joblib import Parallel, delayed


def load_model(model_path, config, device):
    model = ILBERT_T(**config["transformer"]).to(device)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    return model



def predict_with_model(model, test_loader, means, stds, device):
    test_pred = []
    with torch.no_grad():
        for datas, _, _, _ in tqdm(test_loader, desc="Processing Batches", leave=False):
            data = [data.to(device) for data in datas]
            with autocast():
                output = model(data)
            output = output * stds + means
            test_pred.append(output)  # 保持为张量
    return torch.cat(test_pred).cpu().numpy()  # 最后转换为 NumPy


def predict_csv(df):
    config_path = "config_final.yaml"
    config = yaml.load(open(config_path, "r", encoding="utf-8"), Loader=yaml.FullLoader)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


    # model_paths = [f'model_weight/toluene/fold_{i}_best_model.pth' for i in range(1, 11)]

    model_paths = [f'model_weight/styrene/fold_{i}_best_model.pth' for i in range(1, 11)]

    task_params = {
        'Methanol': (0.7419, 0.8531),   # tensor(0.7419) tensor(0.8531) torch.Size([1188, 1, 1])
        'Styren': (0.8531, 0.8531)}   # tensor(0.8747) tensor(0.8263) torch.Size([394, 1, 1])

    means, stds = task_params[config['task']]
    outputs = []

    # config = yaml.load(open(config_path, "r", encoding="utf-8"), Loader=yaml.FullLoader)
    test_dataset = SMILES_dataset(df, tokenizer=SMILES_Atomwise_Tokenizer('vocab.txt'),target='T/K')
    test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False, pin_memory=True)


    for model_path in tqdm(model_paths, desc="Processing Models"):

        best_model = ILBERT_T(**config["transformer"]).to(device)
        best_model.load_state_dict(torch.load(model_path))
        best_model.eval()
        
        test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)
        test_pred = []

        with torch.no_grad():
            for datas, _, _, _ in tqdm(test_loader, desc="Processing Batches", leave=False):
                data = [data.to(device) for data in datas]

                output = best_model(data)
                output = output * stds + means
                test_pred.extend(output.detach().cpu().numpy())

        outputs.append(test_pred)

    outputs = np.array(outputs)
    mean_outputs = np.mean(outputs, axis=0).flatten()
    std_outputs = np.std(outputs, axis=0).flatten()

    return mean_outputs, std_outputs



df=pd.read_csv("commercial_available_IL.csv")
pred,stds= predict_csv(df)

df['Styrene_298'] = pred
df['Styrene_298_STD'] = stds
df.to_csv('commercial_available_IL.csv')

Processing Models: 100%|██████████| 10/10 [00:00<00:00, 12.93it/s]
